In [ ]:
import os
import pandas as pd
from openai import OpenAI
import random

# Set OPENAI_API_KEY as an environment variable before running.
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Set DATA_DIR to the folder holding the BERTopic outputs (default: current
# directory; on Google Colab this is often "/content").
DATA_DIR = os.environ.get("DATA_DIR", ".")

# 載入資料
df = pd.read_csv(os.path.join(DATA_DIR, "topic_info_60_105_7.csv"))  # 包含 Topic、Representation、Representative_Docs 欄位
df_topic = pd.read_csv(os.path.join(DATA_DIR, "document_topic_comparison_60_105_7.csv"))

results = []

for _, row in df.iterrows():
    cluster_id = row["Topic"]
    keywords = row["Representation"]
    # ref_text = row["Representative_Docs"]

    # 抽樣該群中的文本
    df_topic_cluster = df_topic[df_topic["bertopic_topic"] == cluster_id]
    cluster_texts = df_topic_cluster["document"].dropna().tolist()
    sampled_texts = random.sample(cluster_texts, min(60, len(cluster_texts)))
    sampled_texts = random.sample(cluster_texts,max(1, int(len(cluster_texts)*0.3)))
    # sampled_texts = random.sample(cluster_texts, len(cluster_texts))
    # 組合抽樣文本為字串
    sampled_text_str = "\n".join(f"- {t.strip()}" for t in sampled_texts)

    # 組合 prompt
    prompt = f"""
    You are an expert in analyzing election-related texts. Below is a set of keywords and sampled English texts collected from social media or news sources.

    Based on the information provided:

    1. Assign a **clear and specific topic name in English** to this group of texts. Avoid vague labels such as “election” or “election-related discussion.”

    2. Since all texts are related to elections, please focus on more specific subtopics such as candidate comparison, policy controversies, media coverage, campaign strategies, etc.

    3. **Assess the topical coherence of this group:** Is this group thematically stable (i.e., most texts clearly relate to a consistent topic)? Or does it contain multiple competing themes or subtopics?

    4. If the group is internally mixed or fragmented, please indicate that it **should be subdivided**, and briefly explain why. Suggest **specific subtopic titles** if applicable.

    5. Summarize the **tone and rhetorical style** commonly observed in this group. This may include whether the tone is aggressive, calm, persuasive, populist, emotional, formal, or conversational. If there is no consistent tone, indicate that.

    6. Describe the **typical context or setting** in which these texts seem to occur. For example: public speeches, campaign rallies, social media posts, news interviews, policy debates, advertisements, etc. If no common setting can be identified, mention that.

    Please answer using the following format:

    Topic Title: <concise and specific topic name>
    Reason for Topic: <brief rationale>
    Topical Coherence: Stable / Mixed
    Should be Subdivided: Yes / No
    Suggested Subtopics (if any): <List subtopic titles or state "Not applicable">
    Common Tone: <brief summary of rhetorical tone, or state "no consistent tone">
    Typical Context: <brief summary of setting or format, or state "no consistent context">
    ---

    Keywords:
    {keywords}
    (If missing, infer from sampled texts)

    Sampled texts from this group:
    {sampled_text_str}
    """
    # prompt = f"""
    # You are an expert in analyzing election-related texts. Below is a set of keywords and sampled English texts collected from social media or news sources.

    # Based on the information provided:

    # 1. Assign a **clear and specific topic name in English** to this group of texts. Avoid vague labels such as “election” or “election-related discussion.”

    # 2. Since all texts are related to elections, please focus on more specific subtopics such as candidate comparison, policy controversies, media coverage, campaign strategies, etc.

    # 3. **Assess the topical coherence of this group:** Is this group thematically stable (i.e., most texts clearly relate to a consistent topic)? Or does it contain multiple competing themes or subtopics?

    # 4. If the group is internally mixed or fragmented, please indicate that it **should be subdivided**, and briefly explain why. Suggest **specific subtopic titles** if applicable.

    # 5. Summarize the **tone and rhetorical style** commonly observed in this group. This may include whether the tone is aggressive, calm, persuasive, populist, emotional, formal, or conversational. If there is no consistent tone, indicate that.

    # 6. Describe the **typical context or setting** in which these texts seem to occur. For example: public speeches, campaign rallies, social media posts, news interviews, policy debates, advertisements, etc. If no common setting can be identified, mention that.

    # 7. Determine whether **personal background narratives** are a **main theme** of this group. These narratives may include stories about the candidate’s upbringing, military or business background, family life, personal motivations, or religious values.

    # Please answer the following:
    # - Is personal background narrative a **main theme** of this group? (Yes / No)
    # - If **Yes**, briefly describe the common types and purposes of these narratives (e.g., building credibility, emotional appeal, signaling outsider status).
    # - If **No**, describe how these narratives appear (e.g., occasionally mentioned, supportive but not central, absent).

    # Please answer using the following format:

    # Topic Title: <concise and specific topic name>
    # Reason for Topic: <brief rationale>
    # Topical Coherence: Stable / Mixed
    # Should be Subdivided: Yes / No
    # Suggested Subtopics (if any): <List subtopic titles or state "Not applicable">
    # Common Tone: <brief summary of rhetorical tone, or state "no consistent tone">
    # Typical Context: <brief summary of setting or format, or state "no consistent context">
    # Personal Background Narratives:
    #   - Main Theme: Yes / No
    #   - If Yes: <summary of narrative types and rhetorical purposes>
    #   - If No: <summary of role, e.g., marginal, occasional, absent>

    # ---

    # Keywords:
    # {keywords}
    # (If missing, infer from sampled texts)

    # Sampled texts from this group:
    # {sampled_text_str}
    # """


    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
              "role": "system",
              "content": "You are a subject matter expert in analyzing election-related texts."
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
    )

    results.append({
        "cluster": cluster_id,
        "output": response.choices[0].message.content
    })

# 儲存結果
output_df = pd.DataFrame(results)
output_df.to_excel(os.path.join(DATA_DIR, "topic_labels_with_split_recommendation_60_105_7_e5.xlsx"), index=False)